<a href="https://colab.research.google.com/github/molluIdontknow/ELE_Algorithm/blob/main/ELEalgo_FPGA_quant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ELEalgo — SOH MLP FPGA 양자화 (5→16→8→1, Q3.4 / Q23.8)

FPGA팀 파라미터에 맞춰 **보드에서 측정 가능한 서브피처 5개**로 MLP를 학습하고, PTQ → QAT → 정수 연산 검증 → `.mem` 전달물 생성까지 한 번에 한다.

| 항목 | FPGA 파라미터 | 의미 |
|---|---|---|
| 구조 | 5 → 16 → 8 → 1 | `N_FEATURES=5, N_HIDDEN1=16, N_HIDDEN2=8` |
| 입력 | int8, 소수 4bit (**Q3.4**) | 값 = 정수/16, 범위 −8 ~ +7.9375 |
| 가중치 | int8, Q3.4 | 정수/16 |
| bias | int32, 소수 8bit (**Q23.8**) | 정수/256 |
| 은닉값 | int8, Q3.4 | ReLU 후 0 ~ +7.9375 |
| 누산기 / 출력 | int32, 소수 8bit | Q3.4 × Q3.4 = .8 → bias(.8) 가산 → 은닉은 >>>4, 출력은 .8 그대로 |

**정수 연산 순서 (FPGA와 동일하게 시뮬레이션):** `acc = Σ x·w + b` → `hid = sat_int8(acc >>> 4)` → `ReLU` → 다음 층. 출력은 `acc`(Q23.8) 그대로 → STM32에서 `SOH = out/256 × 10 + 90`.

**데이터:** 지난주 통합 서브피처 표 "서브피처 값" 시트 — 60셀 360 RPT (10/25/45/60°C 전부). 라벨 누설 피처(0.2C 총 충전 용량)는 후보에서 제외.

실행 시간 CPU 약 8분.


# §0. 핵심 코드 (고정소수점 정의 · fake-quant · 학습 · 정수 시뮬레이션)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ELEalgo — SOH MLP 고정소수점 양자화 핵심 코드 (FPGA 스펙 기준)
#
# FPGA 파라미터 (FPGA팀 공지):
#   N_FEATURES=5, N_HIDDEN1=16, N_HIDDEN2=8, N_OUTPUTS=1
#   FEATURE_W=8  FEATURE_FRAC=4   → 입력   Q3.4  (int8,  값 = q/16,  범위 −8 ~ +7.9375)
#   WEIGHT_W=8   WEIGHT_FRAC=4    → 가중치 Q3.4  (int8)
#   BIAS_W=32    BIAS_FRAC=8      → bias   Q23.8 (int32, 값 = q/256)
#   HIDDEN_W=8   HIDDEN_FRAC=4    → 은닉값 Q3.4  (int8, ReLU 후 0 ~ +7.9375)
#   ACC_W=32                      → 누산기 int32 (.8 스케일: Q3.4 × Q3.4 = .8)
#   OUTPUT_W=32  OUTPUT_FRAC=8    → 출력   Q23.8 (int32)
#
# FPGA 정수 연산 순서 (문서 7단계):  곱셈 → 누산 → bias 가산 → >>>4 (버림) → 포화 → ReLU
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, random, copy, warnings
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings("ignore")
torch.set_num_threads(2)

FPGA = dict(N_FEATURES=5, N_HIDDEN1=16, N_HIDDEN2=8, N_OUTPUTS=1,
            FEATURE_W=8, WEIGHT_W=8, BIAS_W=32, HIDDEN_W=8, ACC_W=32, OUTPUT_W=32,
            FEATURE_FRAC=4, WEIGHT_FRAC=4, BIAS_FRAC=8, HIDDEN_FRAC=4, OUTPUT_FRAC=8)
FS  = 2 ** FPGA["FEATURE_FRAC"]      # 16  : 입력 1.0 = 정수 16
WS  = 2 ** FPGA["WEIGHT_FRAC"]       # 16
BS  = 2 ** FPGA["BIAS_FRAC"]         # 256
HS  = 2 ** FPGA["HIDDEN_FRAC"]       # 16
OS_ = 2 ** FPGA["OUTPUT_FRAC"]       # 256
SHIFT = FPGA["FEATURE_FRAC"] + FPGA["WEIGHT_FRAC"] - FPGA["HIDDEN_FRAC"]   # 4 : 곱(.8) → 은닉(.4)
assert FPGA["FEATURE_FRAC"] == FPGA["HIDDEN_FRAC"], "입력과 은닉 소수부가 같아야 모든 층이 같은 shift/bias 포맷을 씀"
assert FPGA["BIAS_FRAC"] == FPGA["FEATURE_FRAC"] + FPGA["WEIGHT_FRAC"] == FPGA["OUTPUT_FRAC"], "bias/출력 소수부 = 입력+가중치 소수부 여야 누산기에 그대로 더할 수 있음"
I8_MIN, I8_MAX = -128, 127

def set_fpga(**kw):
    """FPGA 파라미터 변경 (예: set_fpga(WEIGHT_FRAC=7, BIAS_FRAC=11, OUTPUT_FRAC=11)). 전역 스케일 상수 갱신."""
    global FS, WS, BS, HS, OS_, SHIFT
    FPGA.update(kw)
    FS, WS, BS, HS, OS_ = (2 ** FPGA[k] for k in ("FEATURE_FRAC", "WEIGHT_FRAC", "BIAS_FRAC", "HIDDEN_FRAC", "OUTPUT_FRAC"))
    SHIFT = FPGA["FEATURE_FRAC"] + FPGA["WEIGHT_FRAC"] - FPGA["HIDDEN_FRAC"]
    assert FPGA["FEATURE_FRAC"] == FPGA["HIDDEN_FRAC"] and FPGA["BIAS_FRAC"] == FPGA["FEATURE_FRAC"] + FPGA["WEIGHT_FRAC"] == FPGA["OUTPUT_FRAC"]
    print("FPGA:", {k: FPGA[k] for k in ("FEATURE_FRAC", "WEIGHT_FRAC", "BIAS_FRAC", "HIDDEN_FRAC", "OUTPUT_FRAC")}, "shift =", SHIFT)
Y_OFFSET, Y_SCALE = 90.0, 10.0       # SOH 정규화: (SOH−90)/10 → 출력 Q23.8 에서 복원

def set_seed(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
def rmse(a, b): return float(np.sqrt(mean_squared_error(a, b)))
def metrics(y, p):
    return {"RMSE": rmse(y, p), "MAE": float(mean_absolute_error(y, p)), "R2": float(r2_score(y, p)),
            "ErrRate_%": float(np.mean(np.abs(y - p) / y) * 100), "MaxAbsErr": float(np.max(np.abs(y - p)))}

# ═══════════════════════ 1. 입력 스케일링 (STM32 담당) ════════════════════════
class InputScaler:
    """학습 데이터의 min/max 로 feature 를 [lo, hi] 에 선형 매핑 후 Q3.4 정수(int8)로 반올림.
    STM32 는 같은 상수(a_i, b_i)로  q = round(a_i * x + b_i)  만 계산하면 됨.
    lo/hi 를 ±1 로 두면 정수 범위 −16~16 (33단계, 문서안),  ±7 로 두면 −112~112 (225단계)."""
    def __init__(self, lo=-1.0, hi=1.0):
        self.lo, self.hi = lo, hi
    def fit(self, X):
        self.xmin, self.xmax = X.min(0).astype(np.float64), X.max(0).astype(np.float64)
        rng = np.where(self.xmax - self.xmin > 0, self.xmax - self.xmin, 1.0)
        self.a = (self.hi - self.lo) / rng * FS                     # 실수 → 정수 기울기
        self.b = (self.lo - self.xmin * (self.hi - self.lo) / rng) * FS   # 절편
        return self
    def to_int(self, X):
        q = np.round(X.astype(np.float64) * self.a + self.b)
        return np.clip(q, I8_MIN, I8_MAX).astype(np.int32)        # int8 포화
    def to_float(self, X):                                          # 모델 입력(실수) = q/16
        return (self.to_int(X) / FS).astype(np.float32)
    def params(self, feature_names):
        return {"features": feature_names, "lo": self.lo, "hi": self.hi,
                "xmin": self.xmin.tolist(), "xmax": self.xmax.tolist(),
                "a_int_per_unit": self.a.tolist(), "b_int": self.b.tolist(),
                "note": "q_int8 = clip(round(a*x + b), -128, 127); model input = q/16"}

# ═══════════════════════ 2. fake-quant (QAT 용, STE) ════════════════════════
class RoundSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x): return torch.round(x)
    @staticmethod
    def backward(ctx, g): return g
class FloorSTE(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x): return torch.floor(x)
    @staticmethod
    def backward(ctx, g): return g

def fq_weight(w):   # Q3.4 : round 후 int8 포화  (export 때 round 하므로 round)
    return torch.clamp(RoundSTE.apply(w * WS), I8_MIN, I8_MAX) / WS
def fq_bias(b):     # Q23.8
    return RoundSTE.apply(b * BS) / BS
def fq_hidden(h):   # FPGA: (acc >>> 4) 버림 → int8 포화 → ReLU   ⇒ floor + clamp[0, 127/16]
    return torch.clamp(FloorSTE.apply(h * HS), 0, I8_MAX) / HS

class QLinear(nn.Linear):
    qat = False
    def forward(self, x):
        if self.qat: return nn.functional.linear(x, fq_weight(self.weight), fq_bias(self.bias))
        return super().forward(x)

class QMLP(nn.Module):
    """5 → 16 → 8 → 1.  qat=True 면 forward 에서 가중치/bias/은닉값을 FPGA 포맷으로 fake-quant."""
    def __init__(self, d_in=5, h1=16, h2=8, qat=False):
        super().__init__()
        self.l1, self.l2, self.l3 = QLinear(d_in, h1), QLinear(h1, h2), QLinear(h2, 1)
        self.set_qat(qat)
    def set_qat(self, flag):
        self.qat = flag
        for l in (self.l1, self.l2, self.l3): l.qat = flag
    def forward(self, x):
        h = torch.relu(self.l1(x)); h = fq_hidden(h) if self.qat else h
        h = torch.relu(self.l2(h)); h = fq_hidden(h) if self.qat else h
        return self.l3(h)

# ═══════════════════════ 3. 학습 ════════════════════════════════════════════
BASE_CFG = {"batch_size": 16, "epochs": 2000, "learning_rate": 3e-3, "weight_decay": 1e-4,
            "patience": 300, "scheduler": "cosine", "interp_aug": 64, "qat": False,
            "init_state": None}     # init_state: float 모델 weight 로 QAT 시작 (fine-tune)

def interp_pairs(g, r):
    pairs = []
    for cid in np.unique(g):
        idx = np.where(g == cid)[0]; idx = idx[np.argsort(r[idx])]
        pairs += list(zip(idx[:-1], idx[1:]))
    return np.array(pairs)

def train_mlp(Xq_tr, y_tr, Xq_va, y_va, g_tr=None, r_tr=None, cfg=None, verbose=False):
    """Xq_* : 이미 Q3.4 로 양자화된 실수 입력 (q/16).  반환 model, hist, best_ep"""
    cfg = {**BASE_CFG, **(cfg or {})}
    Xtr = torch.tensor(Xq_tr, dtype=torch.float32); ytr = torch.tensor((y_tr - Y_OFFSET) / Y_SCALE, dtype=torch.float32).unsqueeze(1)
    Xva = torch.tensor(Xq_va, dtype=torch.float32); yva = torch.tensor((y_va - Y_OFFSET) / Y_SCALE, dtype=torch.float32).unsqueeze(1)
    pairs = interp_pairs(g_tr, r_tr) if (cfg["interp_aug"] > 0 and g_tr is not None) else None
    model = QMLP(Xq_tr.shape[1], qat=cfg["qat"])
    if cfg["init_state"] is not None: model.load_state_dict(cfg["init_state"])
    crit = nn.MSELoss()
    opt = optim.Adam(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    sched = (optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=200, eta_min=cfg["learning_rate"] * 0.01)
             if cfg["scheduler"] == "cosine" else optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=30))
    best, best_state, best_ep, wait = float("inf"), None, 0, 0
    hist = {"train": [], "val": []}
    for ep in range(1, cfg["epochs"] + 1):
        if pairs is not None:
            k = cfg["interp_aug"]; sel = pairs[np.random.randint(len(pairs), size=k)]; a = torch.rand(k, 1)
            Xa = a * Xtr[sel[:, 0]] + (1 - a) * Xtr[sel[:, 1]]
            if cfg["qat"]: Xa = torch.round(Xa * FS) / FS          # 보간된 입력도 Q3.4 격자에 올림
            ya = a * ytr[sel[:, 0]] + (1 - a) * ytr[sel[:, 1]]
            Xe, ye = torch.cat([Xtr, Xa]), torch.cat([ytr, ya])
        else: Xe, ye = Xtr, ytr
        perm = torch.randperm(len(Xe)); model.train(); tl = 0; nb = 0
        for s in range(0, len(Xe), cfg["batch_size"]):
            b = perm[s:s + cfg["batch_size"]]
            opt.zero_grad(); loss = crit(model(Xe[b]), ye[b]); loss.backward(); opt.step(); tl += loss.item(); nb += 1
        model.eval()
        with torch.no_grad(): vl = crit(model(Xva), yva).item()
        sched.step() if cfg["scheduler"] == "cosine" else sched.step(vl)
        hist["train"].append(tl / nb); hist["val"].append(vl)
        if vl < best - 1e-7: best, best_ep, wait, best_state = vl, ep, 0, copy.deepcopy(model.state_dict())
        else: wait += 1
        if verbose and ep % 100 == 0: print(f"  ep {ep:4d} train {tl/nb:.4f} val {vl:.4f}")
        if wait >= cfg["patience"]: break
    model.load_state_dict(best_state)
    return model, hist, best_ep

def predict_float(model, Xq):
    model.eval()
    with torch.no_grad(): return model(torch.tensor(Xq, dtype=torch.float32)).numpy().flatten() * Y_SCALE + Y_OFFSET

# ═══════════════════════ 4. 정수 가중치 추출 + FPGA 비트 단위 시뮬레이션 ══════════
def export_int(model):
    """float weight → FPGA 정수.  weight: int8 Q3.4,  bias: int32 Q23.8"""
    sd = {k: v.detach().numpy() for k, v in model.state_dict().items()}
    P = {}
    for l in ("l1", "l2", "l3"):
        P[l + "_w"] = np.clip(np.round(sd[l + ".weight"] * WS), I8_MIN, I8_MAX).astype(np.int64)
        P[l + "_b"] = np.round(sd[l + ".bias"] * BS).astype(np.int64)
    return P

def fpga_forward(P, Xq_int, trace=False):
    """FPGA 와 동일한 정수 연산.  Xq_int: (N,5) int8 값.  반환 out_int32 (Q23.8) [, 은닉값 trace]
       acc = Σ x·w (.8)  + bias (.8)   → int32
       hid = sat_int8( acc >> SHIFT )  → ReLU        (>> 는 산술 시프트 = floor, SHIFT = F_FRAC+W_FRAC−H_FRAC = 4)
       out = acc (.8) 그대로 int32"""
    X = Xq_int.astype(np.int64); tr = {}
    def layer(x, w, b, last=False):
        acc = x @ w.T + b                                             # (N, out)  .8 스케일, int64 로 계산 (int32 범위 체크는 아래)
        assert np.abs(acc).max() < 2 ** (FPGA["ACC_W"] - 1), "ACC overflow!"
        if last: return acc, acc
        h = np.clip(acc >> SHIFT, I8_MIN, I8_MAX)                     # >>>SHIFT (floor) → int8 포화
        h = np.maximum(h, 0)                                          # ReLU
        return h, acc
    h1, a1 = layer(X, P["l1_w"], P["l1_b"]); h2, a2 = layer(h1, P["l2_w"], P["l2_b"]); out, a3 = layer(h2, P["l3_w"], P["l3_b"], last=True)
    if trace: return out.flatten(), {"acc1": a1, "h1": h1, "acc2": a2, "h2": h2, "acc3": a3}
    return out.flatten()

def fpga_to_soh(out_int): return out_int / OS_ * Y_SCALE + Y_OFFSET   # Q23.8 → 정규화 SOH → %

def write_mem(arr, path, width):
    """Verilog $readmemh 용 .mem  (2의 보수, 한 줄에 하나, 행 우선 평탄화)"""
    mask = (1 << width) - 1
    with open(path, "w") as f:
        for v in np.asarray(arr).flatten(): f.write(f"{int(v) & mask:0{width // 4}x}\n")


# §1. 데이터 & Feature 5개

**선정 방법:** 15개 선정 서브피처 중 누설 피처(총 충전 용량) 제외한 14개에서, 5개 조합 2002개 전부를 셀 단위 5-fold CV(Ridge)로 평가해 최고 조합 선택. 물리 계열이 겹치지 않게 (충전 전압·충전 시간·방전 전압·저항·ΔSOC) 들어감.

| # | Feature | 계열 | ρ_SOH | 보드 측정 |
|---|---|---|---|---|
| x0 | 0.2C 충전 CC 구간 평균 전압 (V) | ccct 대용 | −0.96 | 0.6A 충전 로그 |
| x1 | 0.2C 충전 3.6→3.8V 통과 시간 (s) | deltat103840 대용 | +0.99 | 0.6A 충전 로그 |
| x2 | 0.2C 방전 중 누적 1.5Ah 시점 전압 (V) | deltat103840 대용 | +0.96 | 0.6A 방전 로그 |
| x3 | 충/방전 전압 갭 저항 SOC 70~90% (Ω) | dcirohm 대용 | −0.91 | 0.6A 충·방전 곡선 |
| x4 | 충/방전 SOC 갭 평균 3.3~4.2V | variance 대용 | −0.94 | 0.6A 충·방전 곡선 |

**순서(x0~x4)는 FPGA·STM32와 공유하는 고정 순서.** 바꾸면 `.mem`과 스케일 상수 전부 다시 생성해야 함.


In [ ]:
XLSX = "/content/drive/MyDrive/subfeatures_60cells.xlsx" if os.path.exists("/content/drive/MyDrive/subfeatures_60cells.xlsx") else "subfeatures_60cells.xlsx"
if not os.path.exists(XLSX):
    from google.colab import drive; drive.mount("/content/drive"); XLSX = "/content/drive/MyDrive/subfeatures_60cells.xlsx"
df = pd.read_excel(XLSX, sheet_name="서브피처 값")

FEATURES = ["[ccct] 0.2C 충전 CC 구간 평균 전압 (선정)",
            "[deltat103840] 0.2C 충전 3.6→3.8V 통과 시간 (선정)",
            "[deltat103840] 0.2C 방전 중 누적 1.5Ah 시점 전압 (선정)",
            "[dcirohm] 충/방전 전압 갭 저항 (SOC 70~90%) (선정)",
            "[variance] 충/방전 SOC 갭 평균 (3.3~4.2V) (선정)"]
SHORT = ["cc_mean_V", "t_chg_3.6-3.8V", "V_dis_1.5Ah", "R_gap_soc70-90", "dSOC_gap_mean"]
X_all = df[FEATURES].values.astype(np.float32)
y_all = df["SOH(%)"].values.astype(np.float32)
g_all = df["cell_id"].values; r_all = df["RPT 파일번호"].values
df["temp"] = df["조건"].str.extract(r"(\d+)°C").astype(int)
print(f"{len(df)} rows, {df.cell_id.nunique()} cells, SOH {y_all.min():.1f}~{y_all.max():.1f}")
print(df["조건"].value_counts().to_string())
pd.DataFrame(X_all, columns=SHORT).describe().T.round(4)

## 1-1. 입력 범위 결정: ±1 (문서안) vs ±7 (전체 범위)

문서 방안은 min-max → −1~1 → ×16 이라 정수가 −16~16, **33단계**뿐. FPGA 파라미터를 안 바꾸고 min-max 를 **−7~+7**로 잡으면 −112~112, 225단계. (±8 을 꽉 채우지 않는 이유: 학습 범위 밖 실측값이 들어와도 ±8에서 포화되기 전 여유를 두기 위해.)

아래에서 두 방식을 같은 fold로 비교한다.


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for i, ax in enumerate(axes):
    for lo_hi, c in [((-1, 1), "tab:red"), ((-7, 7), "tab:blue")]:
        q = InputScaler(*lo_hi).fit(X_all).to_int(X_all)[:, i]
        ax.hist(q, bins=60, alpha=.5, color=c, label=f"range ±{lo_hi[1]}: {len(np.unique(q))} levels")
    ax.set_title(SHORT[i], fontsize=9); ax.set_xlabel("int8 value"); ax.legend(fontsize=7)
fig.suptitle("Quantized input histograms (Q3.4 integer)"); plt.tight_layout(); plt.show()

# §2. 분할 & CV: float → PTQ → QAT

- Holdout: 셀 20%(12셀) — 마지막에 한 번만.
- Dev 48셀 → GroupKFold 5.
- fold마다: ① float 학습 → ② PTQ(float 가중치를 그냥 반올림해서 정수 연산) → ③ QAT(float 가중치로 초기화, fake-quant + STE 로 재학습, lr 1e-3) → ④ QAT 모델을 정수로 내보내 FPGA 정수 연산으로 재평가. ③과 ④가 같아야 "fake-quant 가 FPGA 와 비트 단위로 일치"한다는 뜻.


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
dev_idx, test_idx = next(gss.split(X_all, y_all, groups=g_all))
FOLDS = [(dev_idx[tr], dev_idx[va]) for tr, va in GroupKFold(5).split(dev_idx, groups=g_all[dev_idx])]
print(f"dev {len(dev_idx)} rows / {len(np.unique(g_all[dev_idx]))} cells,  holdout {len(test_idx)} rows / {len(np.unique(g_all[test_idx]))} cells")
print("holdout 셀 조건:", df.iloc[test_idx].groupby("cell_id")["조건"].first().value_counts().to_dict())

def cv_pipeline(lo_hi, seeds=(0,), qat_lr=1e-3, label=""):
    res = {"float": [], "PTQ": [], "QAT": [], "QAT_int": []}; oof = {k: np.full(len(y_all), np.nan) for k in res}
    for s in seeds:
        for k, (tr, va) in enumerate(FOLDS, 1):
            sc = InputScaler(*lo_hi).fit(X_all[tr])
            Xq_tr, Xq_va, Xi_va = sc.to_float(X_all[tr]), sc.to_float(X_all[va]), sc.to_int(X_all[va])
            set_seed(100 * s + k)
            m, _, _ = train_mlp(Xq_tr, y_all[tr], Xq_va, y_all[va], g_all[tr], r_all[tr])
            p = {"float": predict_float(m, Xq_va), "PTQ": fpga_to_soh(fpga_forward(export_int(m), Xi_va))}
            set_seed(100 * s + k)
            mq, _, _ = train_mlp(Xq_tr, y_all[tr], Xq_va, y_all[va], g_all[tr], r_all[tr],
                                 cfg={"qat": True, "init_state": m.state_dict(), "learning_rate": qat_lr})
            p["QAT"] = predict_float(mq, Xq_va); p["QAT_int"] = fpga_to_soh(fpga_forward(export_int(mq), Xi_va))
            for kk in res: res[kk].append(rmse(y_all[va], p[kk])); oof[kk][va] = p[kk]
            print(f"  [{label}] fold{k}: " + "  ".join(f"{kk} {res[kk][-1]:.3f}" for kk in res) + f"   |QAT fake − int| max {np.abs(p['QAT'] - p['QAT_int']).max():.5f}")
    print(f"== {label}: " + "  ".join(f"{kk} {np.mean(v):.3f}±{np.std(v):.3f}" for kk, v in res.items()))
    return res, oof

res_pm1, oof_pm1 = cv_pipeline((-1, 1), label="input ±1")
res_pm7, oof_pm7 = cv_pipeline((-7, 7), label="input ±7")

In [ ]:
# ── 비교 그래프 ──
fig, ax = plt.subplots(figsize=(9, 4)); w = .38; xs = np.arange(4); keys = list(res_pm1)
for off, (res, lab, c) in [(-w/2, (res_pm1, "input ±1 (33 levels)", "tab:red")), (w/2, (res_pm7, "input ±7 (225 levels)", "tab:blue"))]:
    m = [np.mean(res[k]) for k in keys]; s = [np.std(res[k]) for k in keys]
    ax.bar(xs + off, m, w, yerr=s, capsize=3, label=lab, color=c, alpha=.8)
    for x, v in zip(xs + off, m): ax.text(x, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(xs); ax.set_xticklabels(keys); ax.set_ylabel("5-fold RMSE (%SOH)"); ax.set_title("float → PTQ → QAT, by input range"); ax.legend(); plt.show()
INPUT_RANGE = (-7, 7)     # ★ 결과 보고 확정. 이후 전부 이 값 사용
print("확정 입력 범위:", INPUT_RANGE)

## 2-1. OOF predicted vs actual (QAT 정수 연산 결과)

In [ ]:
oof = oof_pm7["QAT_int"]; m_ = ~np.isnan(oof)
conds = sorted(df["조건"].unique()); cc = dict(zip(conds, plt.cm.tab10.colors))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for c in conds:
    k = m_ & (df["조건"] == c).values
    axes[0].scatter(y_all[k], oof[k], s=14, alpha=.7, color=cc[c], label=c)
lo, hi = y_all.min() - 1, y_all.max() + 1
axes[0].plot([lo, hi], [lo, hi], "--", color="gray"); axes[0].fill_between([lo, hi], [lo-3, hi-3], [lo+3, hi+3], color="gray", alpha=.08)
axes[0].set_xlabel("actual SOH (%)"); axes[0].set_ylabel("predicted (FPGA integer sim)"); axes[0].set_title(f"OOF, 48 dev cells — RMSE {rmse(y_all[m_], oof[m_]):.3f}"); axes[0].legend(fontsize=7)
r = pd.DataFrame({"cond": df["조건"][m_], "err2": (y_all[m_] - oof[m_])**2}).groupby("cond")["err2"].mean().pow(.5)
r.plot.bar(ax=axes[1], color=[cc[c] for c in r.index]); axes[1].set_ylabel("RMSE (%)"); axes[1].set_title("OOF RMSE by condition"); axes[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

# §3. Holdout 12셀 — 5 seed

dev 48셀로 학습(그 안에서 20% 를 early-stopping val로), holdout 12셀 평가. seed 5개로 평균±표준편차.


In [ ]:
def holdout_run(seed, lo_hi=INPUT_RANGE):
    gss2 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=1000 + seed)
    tr_, va_ = next(gss2.split(dev_idx, groups=g_all[dev_idx])); tr, va = dev_idx[tr_], dev_idx[va_]
    sc = InputScaler(*lo_hi).fit(X_all[tr])
    set_seed(seed); m, _, _ = train_mlp(sc.to_float(X_all[tr]), y_all[tr], sc.to_float(X_all[va]), y_all[va], g_all[tr], r_all[tr])
    set_seed(seed); mq, hist, ep = train_mlp(sc.to_float(X_all[tr]), y_all[tr], sc.to_float(X_all[va]), y_all[va], g_all[tr], r_all[tr],
                                             cfg={"qat": True, "init_state": m.state_dict(), "learning_rate": 1e-3})
    Xi = sc.to_int(X_all[test_idx])
    return {"float": metrics(y_all[test_idx], predict_float(m, sc.to_float(X_all[test_idx]))),
            "PTQ": metrics(y_all[test_idx], fpga_to_soh(fpga_forward(export_int(m), Xi))),
            "QAT_int": metrics(y_all[test_idx], fpga_to_soh(fpga_forward(export_int(mq), Xi)))}, (m, mq, sc, hist, ep)

ho = [holdout_run(s)[0] for s in range(5)]
tbl = pd.concat({k: pd.DataFrame([h[k] for h in ho]).agg(["mean", "std"]).T for k in ho[0]}, axis=1)
print("Holdout 12 cells, 5 seeds:"); display(tbl.round(3))

# §4. 최종 모델 (QAT) & FPGA 전달물

seed 42 로 최종 학습 → 정수 추출 → 검증 → 파일 생성.


In [ ]:
_, (m_final, mq_final, SC, hist_q, best_ep) = holdout_run(42)
P = export_int(mq_final)
Xi_test = SC.to_int(X_all[test_idx]); Xq_test = SC.to_float(X_all[test_idx])
p_fake = predict_float(mq_final, Xq_test)
out_int, trace = fpga_forward(P, Xi_test, trace=True); p_int = fpga_to_soh(out_int)
print("fake-quant vs 정수연산 최대 차이:", np.abs(p_fake - p_int).max(), " (0 이어야 함)")
print("final holdout:", {k: round(v, 3) for k, v in metrics(y_all[test_idx], p_int).items()})

# ── 자원/범위 점검 ──
for l in ("l1", "l2", "l3"):
    w = P[l + "_w"]; print(f"{l}: weight int8 범위 [{w.min():4d}, {w.max():4d}]  (포화 {np.mean(np.abs(w) == 127)*100:.1f}%)   bias int32 범위 [{P[l+'_b'].min()}, {P[l+'_b'].max()}]")
print(f"acc 최대 |값| = {max(np.abs(trace['acc1']).max(), np.abs(trace['acc2']).max(), np.abs(trace['acc3']).max())}  (int32 한계 {2**31-1})")
print(f"은닉값 포화(=127) 비율: h1 {np.mean(trace['h1']==127)*100:.2f}%  h2 {np.mean(trace['h2']==127)*100:.2f}%   /  0(dead) 비율: h1 {np.mean(trace['h1']==0)*100:.1f}%  h2 {np.mean(trace['h2']==0)*100:.1f}%")
sat_in = np.mean((Xi_test == 127) | (Xi_test == -128)) * 100; print(f"holdout 입력 포화 비율: {sat_in:.2f}%")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(hist_q["train"], label="train"); axes[0].plot(hist_q["val"], label="val"); axes[0].axvline(best_ep, color="gray", ls=":"); axes[0].set_yscale("log"); axes[0].set_title("QAT fine-tune loss"); axes[0].legend()
axes[1].hist(np.concatenate([P[l+"_w"].flatten() for l in ("l1","l2","l3")]), bins=64, color="steelblue"); axes[1].set_title("int8 weight distribution (Q3.4)"); axes[1].set_xlabel("integer value")
for c in conds:
    k = (df["조건"].values[test_idx] == c)
    if k.any(): axes[2].scatter(y_all[test_idx][k], p_int[k], s=22, color=cc[c], label=c)
axes[2].plot([lo, hi], [lo, hi], "--", color="gray"); axes[2].set_xlabel("actual"); axes[2].set_ylabel("FPGA integer prediction"); axes[2].set_title(f"Holdout — RMSE {rmse(y_all[test_idx], p_int):.3f}"); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.show()

## 4-1. 전달 파일 생성

| 파일 | 내용 | 받는 팀 |
|---|---|---|
| `w1.mem` (16×5), `w2.mem` (8×16), `w3.mem` (1×8) | int8 가중치, 2의 보수 hex, 행 우선 (out, in) | FPGA |
| `b1.mem` (16), `b2.mem` (8), `b3.mem` (1) | int32 bias Q23.8, hex | FPGA |
| `golden_vectors.csv` | 정수 입력 x0~x4 → h1[16] → h2[8] → out_int32 → SOH | FPGA (시뮬 비교) |
| `input_scale.json` / `input_scale.h` | feature 순서, a·b 상수: `q = clip(round(a·x + b), −128, 127)` | STM32 |
| `model_int.json` | 위 전부 + 파라미터 한 파일에 | 공용 |


In [ ]:
OUT = "fpga_export"; os.makedirs(OUT, exist_ok=True)
write_mem(P["l1_w"], f"{OUT}/w1.mem", 8); write_mem(P["l2_w"], f"{OUT}/w2.mem", 8); write_mem(P["l3_w"], f"{OUT}/w3.mem", 8)
write_mem(P["l1_b"], f"{OUT}/b1.mem", 32); write_mem(P["l2_b"], f"{OUT}/b2.mem", 32); write_mem(P["l3_b"], f"{OUT}/b3.mem", 32)

# golden vectors: 입력 정수 → 각 층 정수 → 출력
gv = pd.DataFrame(Xi_test, columns=[f"x{i}_int8" for i in range(5)])
for j in range(16): gv[f"h1_{j}"] = trace["h1"][:, j]
for j in range(8):  gv[f"h2_{j}"] = trace["h2"][:, j]
gv["out_int32_Q23_8"] = out_int; gv["soh_pred"] = p_int; gv["soh_actual"] = y_all[test_idx]
gv["cell_id"] = g_all[test_idx]; gv["cond"] = df["조건"].values[test_idx]
gv.to_csv(f"{OUT}/golden_vectors.csv", index=False)

scale = SC.params(SHORT); scale["feature_columns_in_xlsx"] = FEATURES
json.dump(scale, open(f"{OUT}/input_scale.json", "w"), indent=2, ensure_ascii=False)
with open(f"{OUT}/input_scale.h", "w") as f:
    f.write("// SOH MLP input scaling (STM32).  q_int8 = clip(round(a*x + b), -128, 127)\n#define N_FEATURES 5\n")
    f.write("// order: " + ", ".join(SHORT) + "\n")
    f.write("static const float IN_A[5] = {" + ", ".join(f"{v:.8g}f" for v in SC.a) + "};\n")
    f.write("static const float IN_B[5] = {" + ", ".join(f"{v:.8g}f" for v in SC.b) + "};\n")
    f.write("// SOH[%] = out_int32 / 256.0f * 10.0f + 90.0f\n")
json.dump({"fpga_params": FPGA, "features": SHORT, "input_scale": scale,
           **{k: v.tolist() for k, v in P.items()}, "soh_decode": "SOH = out_int32/256*10+90",
           "holdout_rmse_int": rmse(y_all[test_idx], p_int)}, open(f"{OUT}/model_int.json", "w"), indent=1)
torch.save(mq_final.state_dict(), f"{OUT}/mlp_qat_float.pth")
print(sorted(os.listdir(OUT)))
print("\nw1.mem 앞 5줄:", open(f"{OUT}/w1.mem").read().split()[:5], "  ← l1 weight[0][0..4] =", P["l1_w"][0].tolist())

## 4-2. 정수 연산 레퍼런스 (STM32 / Verilog 테스트벤치 대조용)

```c
// FPGA 와 동일한 정수 추론 (C, int32 누산). 입력 x[5] 는 int8 (Q3.4)
int32_t acc; int8_t h1[16], h2[8];
for (j=0;j<16;j++){ acc=b1[j]; for(i=0;i<5;i++) acc+=(int32_t)x[i]*w1[j][i];
                    acc>>=4;  acc = acc>127?127:(acc<-128?-128:acc);  h1[j]= acc<0?0:acc; }
for (k=0;k<8;k++) { acc=b2[k]; for(j=0;j<16;j++) acc+=(int32_t)h1[j]*w2[k][j];
                    acc>>=4;  acc = acc>127?127:(acc<-128?-128:acc);  h2[k]= acc<0?0:acc; }
acc=b3[0]; for(k=0;k<8;k++) acc+=(int32_t)h2[k]*w3[0][k];      // out = acc (Q23.8)
float soh = acc/256.0f*10.0f + 90.0f;
```
`>>` 는 **산술 시프트(버림)**. 반올림으로 바꾸면 golden vector 와 안 맞음.


# §6. 제안: WEIGHT_FRAC 4 → 7 (가중치 Q0.7)

§4 점검에서 int8 가중치가 **−8~+6 범위만 씀** (256단계 중 15단계). 입력을 ±7로 키우니 가중치가 ±0.5 안으로 작아져서, Q3.4(0.0625 간격)로는 가중치 해상도가 너무 거칠다. 그래서 PTQ 가 1.0 까지 나빠지고 QAT 로 겨우 복구하는 것.

**같은 8bit 에서 소수부만 7bit 로 바꾸면** (가중치 범위 ±0.99, 간격 0.0078):
- `WEIGHT_FRAC = 7`, `BIAS_FRAC = OUTPUT_FRAC = 11` (= FEATURE_FRAC + WEIGHT_FRAC), 은닉 shift `>>> 7`
- 비트폭(8/8/32/8/32)과 구조는 그대로. RTL 에서 바뀌는 건 shift 상수와 bias/출력 소수점 위치뿐.
- SOH 복원: `out/2048 × 10 + 90`

아래에서 같은 fold 로 비교하고, 채택될 경우를 대비해 `fpga_export_wfrac7/` 에 전달물도 따로 만든다.


In [ ]:
set_fpga(WEIGHT_FRAC=7, BIAS_FRAC=11, OUTPUT_FRAC=11)
res_w7, oof_w7 = cv_pipeline(INPUT_RANGE, label="W_FRAC=7")
cmp = pd.DataFrame({"W_FRAC=4 (FPGA spec)": {k: f"{np.mean(v):.3f} ± {np.std(v):.3f}" for k, v in res_pm7.items()},
                    "W_FRAC=7 (proposal)":  {k: f"{np.mean(v):.3f} ± {np.std(v):.3f}" for k, v in res_w7.items()}})
print("5-fold CV RMSE (input ±7):"); display(cmp)

# holdout + export (variant)
_, (m7, mq7, SC7, _, _) = holdout_run(42)
P7 = export_int(mq7); Xi7 = SC7.to_int(X_all[test_idx]); out7, tr7 = fpga_forward(P7, Xi7, trace=True); p7 = fpga_to_soh(out7)
print("variant holdout:", {k: round(v, 3) for k, v in metrics(y_all[test_idx], p7).items()}, " | fake-int diff", np.abs(predict_float(mq7, SC7.to_float(X_all[test_idx])) - p7).max())
for l in ("l1", "l2", "l3"): print(f"  {l}: weight int8 범위 [{P7[l+'_w'].min()}, {P7[l+'_w'].max()}]  포화 {np.mean(np.abs(P7[l+'_w'])==127)*100:.1f}%")
print("  acc 최대 |값| =", max(np.abs(tr7[k]).max() for k in ("acc1","acc2","acc3")))
OUT7 = "fpga_export_wfrac7"; os.makedirs(OUT7, exist_ok=True)
for l, n in zip(("l1","l2","l3"), (1,2,3)):
    write_mem(P7[l+"_w"], f"{OUT7}/w{n}.mem", 8); write_mem(P7[l+"_b"], f"{OUT7}/b{n}.mem", 32)
gv7 = pd.DataFrame(Xi7, columns=[f"x{i}_int8" for i in range(5)])
for j in range(16): gv7[f"h1_{j}"] = tr7["h1"][:, j]
for j in range(8):  gv7[f"h2_{j}"] = tr7["h2"][:, j]
gv7["out_int32_Q21_11"] = out7; gv7["soh_pred"] = p7; gv7["soh_actual"] = y_all[test_idx]; gv7["cell_id"] = g_all[test_idx]
gv7.to_csv(f"{OUT7}/golden_vectors.csv", index=False)
sc7 = SC7.params(SHORT); json.dump(sc7, open(f"{OUT7}/input_scale.json", "w"), indent=2, ensure_ascii=False)
with open(f"{OUT7}/input_scale.h", "w") as f:
    f.write("// VARIANT: WEIGHT_FRAC=7, BIAS_FRAC=OUTPUT_FRAC=11, hidden shift >>>7\\n#define N_FEATURES 5\\n// order: " + ", ".join(SHORT) + "\\n")
    f.write("static const float IN_A[5] = {" + ", ".join(f"{v:.8g}f" for v in SC7.a) + "};\\nstatic const float IN_B[5] = {" + ", ".join(f"{v:.8g}f" for v in SC7.b) + "};\\n// SOH[%] = out_int32 / 2048.0f * 10.0f + 90.0f\\n")
json.dump({"fpga_params": FPGA, "features": SHORT, "input_scale": sc7, **{k: v.tolist() for k, v in P7.items()}, "soh_decode": "SOH = out_int32/2048*10+90",
           "holdout_rmse_int": rmse(y_all[test_idx], p7)}, open(f"{OUT7}/model_int.json", "w"), indent=1)
print(sorted(os.listdir(OUT7)))
set_fpga(WEIGHT_FRAC=4, BIAS_FRAC=8, OUTPUT_FRAC=8)   # 원래 스펙으로 복귀

# §5. 요약 / 전달 사항

1. **입력 스케일 −7~+7** 권장 (문서안 ±1 대비 RMSE 개선, FPGA 파라미터 변경 없음). STM32 가 `input_scale.h` 상수로 `q = round(a·x+b)` 만 계산.
2. **PTQ 는 부족, QAT 필요** — Q3.4 가중치(0.0625 간격) + 버림 때문. QAT 후 float 대비 손실 거의 없음.
3. fake-quant 학습 결과와 정수 시뮬레이션이 **비트 단위로 일치** (차이 0). `golden_vectors.csv` 로 RTL 시뮬 대조.
4. 누산기 int32 여유 충분 (최대 |acc| ≪ 2³¹). 은닉값 포화 거의 없음.
5. **제안 (§6)**: `WEIGHT_FRAC=7, BIAS_FRAC=OUTPUT_FRAC=11, shift>>>7` 로 바꾸면 PTQ 만으로도 쓸 만해지고 QAT 도 더 좋아짐. 비트폭 변경 없음. 채택 시 `fpga_export_wfrac7/` 사용.
6. feature 5개 순서 고정: `cc_mean_V, t_chg_3.6-3.8V, V_dis_1.5Ah, R_gap_soc70-90, dSOC_gap_mean`. 바뀌면 전부 재생성.
7. 미결: dcirohm 대용이 xlsx(전압 갭 저항) vs docx(1.5A 펄스) 로 갈려 있음. 펄스로 가면 x3 교체 후 재학습.
